<a href="https://colab.research.google.com/github/tustus1022-ui/esaa/blob/main/%EC%B5%9C%EC%A2%85_neutralization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

val_final 파일 만들기(저장하기)

In [ ]:
import os
from google.colab import drive
import pandas as pd

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install numerapi -q
import numerapi
import getpass

NUMERAI_PUBLIC_ID = getpass.getpass("Public ID 입력: ").strip()
NUMERAI_SECRET_KEY = getpass.getpass("Secret Key 입력: ").strip()

napi = numerapi.NumerAPI(public_id=NUMERAI_PUBLIC_ID, secret_key=NUMERAI_SECRET_KEY)
print(napi.get_models())

Public ID 입력: ··········
Secret Key 입력: ··········
{'esaa_maddox': '37dd8d41-54d3-4e4a-a6e8-7b5c7a49ad0f', 'esaa': 'b334bb14-b052-444e-a266-7bb8619e749f'}


In [ ]:
import os
os.makedirs('/content/numerai_data', exist_ok=True)

if not os.path.exists("/content/numerai_data/validation.parquet"):
    napi.download_dataset("v5.2/validation.parquet", "/content/numerai_data/validation.parquet")

print("다운로드 완료")

/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'numerai-datasets-us-west-2.s3-accelerate.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(

/content/numerai_data/validation.parquet:   0%|          | 0.00/4.34G [00:00<?, ?B/s]
/content/numerai_data/validation.parquet:  57%|█████▋    | 2.49G/4.34G [01:26<00:51, 36.3MB/s]
/content/numerai_data/validation.parquet:  66%|██████▋   | 2.88G/4.34G [00:11<00:00, 4.26GB/s]
/content/numerai_data/validation.parquet:  66%|██████▋   | 2.88G/4.34G [00:11<00:07, 185MB/s] 
/content/numerai_data/validation.parquet:  66%|██████▋   | 2.89G/4.34G [00:11<00:07, 183MB/s]
/content/numerai_data/validation.parquet:  71%|███████   | 3.06G/4.34G [00:16<00:11, 115MB/s]
/content/numerai_data/validation.parquet:  73%|███████▎  | 3.16G/4.34G [00:19<00:12, 95.1M

다운로드 완료


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SAVE_DIR = "/content/drive/MyDrive/ESAA/numerai_postprocess"

with open(f"{SAVE_DIR}/selected_features.json") as f:
    selected_features = json.load(f)

scaler = joblib.load(f"{SAVE_DIR}/fitted_scaler.joblib")
pca = joblib.load(f"{SAVE_DIR}/fitted_pca.joblib")

def iter_parquet_batches(path, columns, batch_size):
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(columns=columns, batch_size=batch_size):
        chunk = batch.to_pandas()
        if chunk.index.name is not None or "id" not in chunk.columns:
            chunk = chunk.reset_index()
        yield chunk

def clean_array(arr):
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def add_era_level_lag_features(df, pca_cols, lags=(1, 2)):
    era_mean = df.groupby("era")[pca_cols].mean().sort_index()
    lag_frames = []
    for lag in lags:
        shifted = era_mean.shift(lag)
        shifted.columns = [f"{c}_lag{lag}" for c in pca_cols]
        lag_frames.append(shifted)
    era_lag_features = pd.concat(lag_frames, axis=1)
    return df.merge(era_lag_features, on="era", how="left")

In [ ]:
VAL_PATH = "/content/numerai_data/validation.parquet"
BATCH_SIZE = 100_000

X_parts, era_parts, id_parts = [], [], []
val_columns = ["id", "era"] + selected_features

for chunk in iter_parquet_batches(VAL_PATH, val_columns, BATCH_SIZE):
    X = clean_array(chunk[selected_features].to_numpy(dtype=np.float32))
    X_scaled = scaler.transform(X)
    X_pca = pca.transform(X_scaled).astype(np.float32)
    X_parts.append(X_pca)
    era_parts.append(chunk["era"].to_numpy())
    id_parts.append(chunk["id"].to_numpy())
    print(f"처리: {sum(len(p) for p in id_parts)} rows")

pca_cols = [f"pca_{i}" for i in range(pca.n_components_)]
val_pca_df = pd.DataFrame(np.vstack(X_parts), columns=pca_cols)
val_pca_df["era"] = np.concatenate(era_parts)
val_pca_df["id"] = np.concatenate(id_parts)

print("val_pca_df shape:", val_pca_df.shape)

처리: 100000 rows
처리: 200000 rows
처리: 300000 rows
처리: 400000 rows
처리: 500000 rows
처리: 600000 rows
처리: 700000 rows
처리: 800000 rows
처리: 900000 rows
처리: 1000000 rows
처리: 1100000 rows
처리: 1200000 rows
처리: 1300000 rows
처리: 1400000 rows
처리: 1500000 rows
처리: 1600000 rows
처리: 1700000 rows
처리: 1800000 rows
처리: 1900000 rows
처리: 2000000 rows
처리: 2100000 rows
처리: 2200000 rows
처리: 2300000 rows
처리: 2400000 rows
처리: 2500000 rows
처리: 2600000 rows
처리: 2700000 rows
처리: 2800000 rows
처리: 2900000 rows
처리: 3000000 rows
처리: 3100000 rows
처리: 3200000 rows
처리: 3300000 rows
처리: 3400000 rows
처리: 3500000 rows
처리: 3600000 rows
처리: 3700000 rows
처리: 3800000 rows
처리: 3900000 rows
처리: 4000000 rows
처리: 4099982 rows
val_pca_df shape: (4099982, 502)


In [ ]:
val_final = add_era_level_lag_features(val_pca_df, pca_cols)

lag_cols = [c for c in val_final.columns if "_lag" in c]
val_final[lag_cols] = val_final[lag_cols].fillna(0)

val_final['era'] = val_final['era'].astype(int)

print("val_final shape:", val_final.shape)  # (4085748, 1502) 나와야 함

val_final shape: (4099982, 1502)


In [ ]:
val_final.to_parquet(f"{SAVE_DIR}/val_final_cache.parquet")
print("저장 완료 — 다음부턴 이 파일만 불러오면 됨")

저장 완료 — 다음부턴 이 파일만 불러오면 됨


In [ ]:
print("era 범위:", val_final['era'].min(), "~", val_final['era'].max())
print("era 개수:", val_final['era'].nunique())

era 범위: 575 ~ 1230
era 개수: 656


In [ ]:
PRED_PATH = "/content/drive/MyDrive/ESAA/numerai_postprocess/ensemble4_all7_without_mlp_rankavg_predictions.csv"  # csv 또는 parquet

if PRED_PATH.endswith(".csv"):
    ensemble4_predictions = pd.read_csv(PRED_PATH)
else:
    ensemble4_predictions = pd.read_parquet(PRED_PATH)

print(f"예측값 파일 shape: {ensemble4_predictions.shape}")
print(ensemble4_predictions.head(3))

예측값 파일 shape: (4085748, 2)
                 id  prediction
0  n000101811a8a843    0.472461
1  n001e1318d5072ac    0.863504
2  n002a9c5ab785cbb    0.456060


In [ ]:
merged = val_final.merge(ensemble4_predictions, on='id', how='inner')
print(merged.shape)
print("era 범위:", merged['era'].min(), "~", merged['era'].max())  # 575~1223 나와야 함

(4085748, 1503)
era 범위: 575 ~ 1228


In [ ]:
merged.to_parquet(f"{SAVE_DIR}/ensemble4_with_features_for_neutralization.parquet")
print("merged 저장 완료 (target 미포함 버전)")

target_df = pd.read_parquet(VAL_PATH, columns=["id", "target"])
print(f"target_df shape: {target_df.shape}")

merged = merged.merge(target_df, on="id", how="left")
print(f"target 추가 후 merged shape: {merged.shape}")
print(f"target 결측치 개수: {merged['target'].isna().sum()}")

merged.to_parquet(f"{SAVE_DIR}/ensemble4_with_features_for_neutralization.parquet")
print("target 포함 최종본 저장 완료")

/content/numerai_data/validation.parquet:  57%|█████▋    | 2.49G/4.34G [22:59<17:09, 1.80MB/s]


merged 저장 완료 (target 미포함 버전)
target_df shape: (4099982, 1)
target 추가 후 merged shape: (4085748, 1504)
target 결측치 개수: 21370
target 포함 최종본 저장 완료


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

FEATURE_COLS = [c for c in merged.columns if c.startswith("pca_") and "_lag" not in c]
print(f"neutralization 기준 feature 개수: {len(FEATURE_COLS)}")  # 500이어야 함


def neutralize_era(pred: np.ndarray, features: np.ndarray, proportion: float) -> np.ndarray:
    """era 하나에 대해 neutralization 적용 (numerai 표준: rank -> gaussian -> 회귀 제거 -> rank)"""
    if proportion == 0:
        return pd.Series(pred).rank(pct=True, method="first").values

    pred_rank = pd.Series(pred).rank(pct=True, method="first")
    pred_rank = pred_rank.clip(1e-6, 1 - 1e-6)
    pred_gauss = norm.ppf(pred_rank)

    exposures = np.linalg.lstsq(features, pred_gauss, rcond=None)[0]
    neutralized = pred_gauss - proportion * (features @ exposures)

    return pd.Series(neutralized).rank(pct=True, method="first").values


def apply_neutralization_all_eras(df: pd.DataFrame, proportion: float) -> np.ndarray:
    """era별로 순회하며 neutralize 적용, 원래 순서 그대로 복원"""
    out = np.empty(len(df), dtype=np.float64)
    for era, group in df.groupby("era"):
        idx = group.index.values
        pred = group["prediction"].values
        feats = group[FEATURE_COLS].values
        out_local = neutralize_era(pred, feats, proportion)
        out[df.index.get_indexer(idx)] = out_local
    return out



neutralization 기준 feature 개수: 500


In [ ]:
def era_wise_corr(df: pd.DataFrame, pred_col: str) -> pd.Series:
    def _corr(g):
        return np.corrcoef(g[pred_col].values, g["target"].values)[0, 1]
    return df.groupby("era").apply(_corr)


def feature_exposure_stats(df: pd.DataFrame, pred_col: str):
    """era별로 pred와 각 feature의 상관을 한 번에 벡터화 계산 -> (mean_fe, max_fe)"""
    mean_exps, max_exps = [], []
    for era, group in df.groupby("era"):
        pred = group[pred_col].values
        pred_c = pred - pred.mean()
        pred_std = pred_c.std()
        if pred_std == 0:
            continue
        feats = group[FEATURE_COLS].values
        feats_c = feats - feats.mean(axis=0)
        feats_std = feats_c.std(axis=0)
        feats_std[feats_std == 0] = 1e-9
        cov = (feats_c * pred_c[:, None]).mean(axis=0)
        corrs = cov / (feats_std * pred_std)

        mean_exps.append(np.nanmean(np.abs(corrs)))
        max_exps.append(np.nanmax(np.abs(corrs)))
    return float(np.mean(mean_exps)), float(np.mean(max_exps))



강도별 비교 루프 실행

In [ ]:
PROPORTIONS = [0.0, 0.3, 0.5, 0.7, 1.0]
results = []

for prop in PROPORTIONS:
    print(f"\n=== neutralization {prop} 진행 중 (era {merged['era'].nunique()}개) ===")
    col_name = f"pred_neut_{prop}"
    merged[col_name] = apply_neutralization_all_eras(merged, prop)

    corrs = era_wise_corr(merged, col_name)
    mean_corr = corrs.mean()
    std_corr = corrs.std()
    sharpe = mean_corr / std_corr if std_corr != 0 else np.nan

    mean_fe, max_fe = feature_exposure_stats(merged, col_name)

    row = {
        "neutralization": prop,
        "mean_corr": round(mean_corr, 4),
        "std_corr": round(std_corr, 4),
        "sharpe": round(sharpe, 4),
        "mean_feature_exposure": round(mean_fe, 4),
        "max_feature_exposure": round(max_fe, 4),
    }
    results.append(row)
    print(row)

results_df = pd.DataFrame(results)
print("\n=== Neutralization 강도별 비교 (ensemble4, era-wise) ===")
print(results_df)



=== neutralization 0.0 진행 중 (era 654개) ===


/tmp/ipykernel_4401/4165764764.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("era").apply(_corr)


{'neutralization': 0.0, 'mean_corr': np.float64(0.0161), 'std_corr': 0.0164, 'sharpe': np.float64(0.9806), 'mean_feature_exposure': 0.0347, 'max_feature_exposure': 0.3298}

=== neutralization 0.3 진행 중 (era 654개) ===


/tmp/ipykernel_4401/4165764764.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("era").apply(_corr)


{'neutralization': 0.3, 'mean_corr': np.float64(0.0157), 'std_corr': 0.0162, 'sharpe': np.float64(0.971), 'mean_feature_exposure': 0.0327, 'max_feature_exposure': 0.3108}

=== neutralization 0.5 진행 중 (era 654개) ===


...

{'neutralization': 0.5, 'mean_corr': np.float64(0.015), 'std_corr': 0.0158, 'sharpe': np.float64(0.9462), 'mean_feature_exposure': 0.0296, 'max_feature_exposure': 0.2822}

=== neutralization 0.7 진행 중 (era 654개) ===


/tmp/ipykernel_4401/4165764764.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("era").apply(_corr)


{'neutralization': 0.7, 'mean_corr': np.float64(0.013), 'std_corr': 0.015, 'sharpe': np.float64(0.8664), 'mean_feature_exposure': 0.023, 'max_feature_exposure': 0.2208}

=== neutralization 1.0 진행 중 (era 654개) ===


/tmp/ipykernel_4401/4165764764.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby("era").apply(_corr)


{'neutralization': 1.0, 'mean_corr': np.float64(0.0046), 'std_corr': 0.0133, 'sharpe': np.float64(0.3478), 'mean_feature_exposure': 0.0022, 'max_feature_exposure': 0.0096}

=== Neutralization 강도별 비교 (ensemble4, era-wise) ===
   neutralization  mean_corr  std_corr  sharpe  mean_feature_exposure  \
0             0.0     0.0161    0.0164  0.9806                 0.0347   
1             0.3     0.0157    0.0162  0.9710                 0.0327   
2             0.5     0.0150    0.0158  0.9462                 0.0296   
3             0.7     0.0130    0.0150  0.8664                 0.0230   
4             1.0     0.0046    0.0133  0.3478                 0.0022   

   max_feature_exposure  
0                0.3298  
1                0.3108  
2                0.2822  
3                0.2208  
4                0.0096  


In [ ]:
results_df.to_csv(f"{SAVE_DIR}/neutralization_comparison_ensemble4.csv", index=False)
print(f"결과 저장 완료: {SAVE_DIR}/neutralization_comparison_ensemble4.csv")

결과 저장 완료: /content/drive/MyDrive/ESAA/numerai_postprocess/neutralization_comparison_ensemble4.csv


In [ ]:
import pandas as pd

SUBMIT_PROPORTIONS = [0.3, 0.5]

submission_paths = {}

for prop in SUBMIT_PROPORTIONS:
    col_name = f"pred_neut_{prop}"
    assert col_name in merged.columns, f"{col_name} 컬럼이 없음 -> cell 15 다시 확인"

    sub_df = merged[["id", col_name]].rename(columns={col_name: "prediction"})


    print(f"[{prop}] prediction 범위: {sub_df['prediction'].min():.6f} ~ {sub_df['prediction'].max():.6f}")

    prop_str = str(prop).replace(".", "")
    save_path = f"{SAVE_DIR}/ensemble4_neutralized_{prop_str}_submission.csv"
    sub_df.to_csv(save_path, index=False)
    submission_paths[prop] = save_path
    print(f"저장 완료: {save_path} (shape: {sub_df.shape})")


[0.3] prediction 범위: 0.000137 ~ 1.000000
저장 완료: /content/drive/MyDrive/ESAA/numerai_postprocess/ensemble4_neutralized_03_submission.csv (shape: (4085748, 2))
[0.5] prediction 범위: 0.000137 ~ 1.000000
저장 완료: /content/drive/MyDrive/ESAA/numerai_postprocess/ensemble4_neutralized_05_submission.csv (shape: (4085748, 2))


In [ ]:
import time

MODEL_ID = napi.get_models()["esaa_maddox"]

diagnostics_results = {}

for prop, path in submission_paths.items():
    print(f"\n=== neutralization {prop} diagnostics 업로드 ===")
    diag_id = napi.upload_diagnostics(path, model_id=MODEL_ID)
    print(f"업로드 완료, diagnostics_id: {diag_id}")


    result = None
    for attempt in range(30):
        raw = napi.diagnostics(model_id=MODEL_ID, diagnostics_id=diag_id)

        if attempt == 0:
            print(f"  raw 타입: {type(raw)}")
            print(f"  raw 내용 (일부): {raw if not isinstance(raw, list) else raw[:2]}")

        if isinstance(raw, list):

            matched = [
                r for r in raw
                if r.get("id") == diag_id or r.get("diagnosticsId") == diag_id
            ]
            result = matched[0] if matched else (raw[0] if raw else {})
        else:
            result = raw

        status = result.get("status")
        print(f"  [{attempt}] status: {status}")
        if status == "done":
            break
        time.sleep(20)

    diagnostics_results[prop] = result
    print(f"neutralization {prop} 결과:")
    print(result)


=== neutralization 0.3 diagnostics 업로드 ===
업로드 완료, diagnostics_id: 2e4bd5d9-eebf-4683-8230-b4a660a5c196
  raw 타입: <class 'list'>
  raw 내용 (일부): [{'message': 'valid', 'validationCorrPlusMmcSharpeDiffRating': None, 'validationCorrV4Sharpe': None, 'validationCorrV4CorrWExamplePreds': None, 'validationCorrPlusMmcMean': None, 'validationFncV4Mean': None, 'validationMmcStdRating': None, 'validationCorrSharpeRating': None, 'status': 'valid', 'validationMmcSharpe': None, 'validationRicCorrWExamplePreds': None, 'validationCorrV4Mean': None, 'validationFncV4MaxDrawdown': None, 'validationRicStd': None, 'validationFeatureNeutralCorrV3Mean': None, 'validationCorrMaxDrawdown': None, 'validationRicMaxDrawdown': None, 'validationRicSharpe': None, 'validationIcV2Std': None, 'validationCorrMeanRating': None, 'validationCorrPlusMmcSharpe': None, 'validationAlphaStd': None, 'validationIcV2CorrWExamplePreds': None, 'erasAcceptedCount': 651, 'validationRicMean': None, 'validationIcV2MaxDrawdown': None, 'v

In [ ]:
summary = []
for prop, result in diagnostics_results.items():
    summary.append({
        "neutralization": prop,
        "official_sharpe": result.get("validationCorrSharpe"),
        "official_feature_exposure": result.get("validationFeatureCorrMax"),
        "official_mean_corr": result.get("validationCorrMean"),
        "official_bmc": result.get("validationBmcMean"),
        "official_max_drawdown": result.get("validationMaxDrawdown"),
        "official_adjusted_sharpe": result.get("validationAdjustedSharpe"),
    })

summary_df = pd.DataFrame(summary)
print("\n=== 공식 Diagnostics 기준 강도별 비교 ===")
print(summary_df)

summary_df.to_csv(f"{SAVE_DIR}/official_diagnostics_comparison.csv", index=False)
print("저장 완료")

summary_df = pd.DataFrame(summary)
print("\n=== 공식 Diagnostics 기준 강도별 비교 ===")
print(summary_df)

summary_df.to_csv(f"{SAVE_DIR}/official_diagnostics_comparison.csv", index=False)
print("저장 완료")


=== 공식 Diagnostics 기준 강도별 비교 ===
   neutralization  official_sharpe  official_feature_exposure  \
0             0.3         0.982595                   0.310963   
1             0.5         0.957875                   0.281338   

   official_mean_corr  official_bmc  official_max_drawdown  \
0            0.015417     -0.002311              -0.172732   
1            0.014790     -0.001918              -0.169219   

   official_adjusted_sharpe  
0                  1.278895  
1                  1.098003  
저장 완료

=== 공식 Diagnostics 기준 강도별 비교 ===
   neutralization  official_sharpe  official_feature_exposure  \
0             0.3         0.982595                   0.310963   
1             0.5         0.957875                   0.281338   

   official_mean_corr  official_bmc  official_max_drawdown  \
0            0.015417     -0.002311              -0.172732   
1            0.014790     -0.001918              -0.169219   

   official_adjusted_sharpe  
0                  1.278895  
1         

neutralize(0.3)으로 하기로 결정

In [ ]:
# validation.parquet에서 마지막 era들의 target이 실제로 채워져 있는지 확인
print(val_final.groupby('era')['id'].count().tail(10))

# live.parquet 다운받아서 거기 era가 뭔지 직접 확인
napi.download_dataset("v5.2/live.parquet", "/content/numerai_data/live.parquet")
live_pf = pq.ParquetFile("/content/numerai_data/live.parquet")
live_sample = live_pf.read_row_group(0, columns=['era']).to_pandas()
print("live era:", live_sample['era'].unique())

era
1221    7079
1222    7105
1223    7102
1224    7061
1225    7076
1226    7057
1227    7161
1228    7152
1229    7138
1230    7096
Name: id, dtype: int64


/content/numerai_data/live.parquet: 100%|██████████| 9.61M/9.61M [00:00<00:00, 9.90MB/s]

live era: ['X']


In [ ]:
val_raw_with_target = pd.read_parquet(
    "/content/numerai_data/validation.parquet",
    columns=["id", "era", "target"]
)
print(val_raw_with_target.groupby('era')['target'].apply(lambda x: x.isna().mean()).tail(10))

era
1221    0.0
1222    0.0
1223    0.0
1224    0.0
1225    0.0
1226    1.0
1227    1.0
1228    1.0
1229    1.0
1230    1.0
Name: target, dtype: float64
